# MedASR Medical Speech Recognition with OpenVINOThis notebook demonstrates converting Google's MedASR (Medical Automatic Speech Recognition) model to OpenVINO format with FP16 and INT8 quantization.**Table of Contents:**1. [Installation](#installation)2. [Login to HuggingFace](#login-huggingface)3. [Load Model](#load-model)4. [Prepare Audio Data](#prepare-audio)5. [PyTorch Inference](#pytorch-inference)6. [Convert to OpenVINO FP16](#convert-fp16)7. [INT8 Quantization](#int8-quantization)8. [Accuracy Comparison](#accuracy-comparison)9. [Performance Benchmarking](#benchmarking)

## 1. Installation <a id="installation"></a>

Install required packages for model conversion and optimization.

In [45]:
%pip install -q "openvino>=2024.4.0" "nncf>=2.13.0" "torch>=2.1" "transformers>=5.4.0" "librosa" "soundfile" "huggingface_hub" "matplotlib" "numpy"

Note: you may need to restart the kernel to use updated packages.


## 2. Login to HuggingFace <a id="login-huggingface"></a>

To run the model, you must be a registered user in 🤗 [Hugging Face Hub](https://huggingface.co/). 

The MedASR model is gated and requires you to:
1. Visit the [MedASR model card](https://huggingface.co/google/medasr)
2. Carefully read the terms of usage
3. Click the accept button to agree to the license

You will need to use an access token for the code below to run. For more information on access tokens, refer to [this section of the documentation](https://huggingface.co/docs/hub/security-tokens).

You can login to Hugging Face Hub using the following code:

In [ ]:
# Login to HuggingFace Hub to get access to the pretrained model

from huggingface_hub import notebook_login, whoami

try:
    whoami()
    print('Authorization token already provided')
except OSError:
    notebook_login()

## 3. Load Model <a id="load-model"></a>

Load Google's MedASR model from HuggingFace. This is a CTC-based ASR model optimized for medical terminology.

In [46]:
from transformers import pipeline
import huggingface_hub
import librosa
import numpy as np
import torch
from pathlib import Path
import time

MODEL_ID = "google/medasr"
print(f"Loading model: {MODEL_ID}")

# Load model using pipeline
pipe = pipeline("automatic-speech-recognition", model=MODEL_ID, trust_remote_code=True)

# Extract model components
model = pipe.model
feature_extractor = pipe.feature_extractor
tokenizer = pipe.tokenizer

print(f"✓ Model loaded: {type(model).__name__}")
print(f"✓ Feature extractor: {type(feature_extractor).__name__}")
print(f"✓ Tokenizer vocab size: {tokenizer.vocab_size}")

Loading model: google/medasr


Loading weights:   0%|          | 0/368 [00:00<?, ?it/s]

✓ Model loaded: LasrForCTC
✓ Feature extractor: LasrFeatureExtractor
✓ Tokenizer vocab size: 512


## 4. Prepare Audio Data <a id="prepare-audio"></a>

Download test audio and prepare it for model conversion. We use **10-second audio** for optimal GPU performance.

- Creates model with shape `[1, 998, 128]`
- Longer audio can be processed via chunking

In [47]:
# Download test audio from HuggingFace
audio_file = huggingface_hub.hf_hub_download('google/medasr', 'test_audio.wav')
speech_full, sample_rate = librosa.load(audio_file, sr=16000)

print(f"Full audio duration: {len(speech_full)/sample_rate:.2f} seconds")

# Use 10s audio for optimal model shape
OPTIMAL_DURATION = 10.0
speech_10s = speech_full[:int(OPTIMAL_DURATION * sample_rate)]

print(f"Optimized audio duration: {len(speech_10s)/sample_rate:.2f} seconds")
print(f"Sample rate: {sample_rate} Hz")

# Extract features for model conversion
inputs = feature_extractor(speech_10s, sampling_rate=sample_rate, return_tensors="pt", 
                           padding=True, return_attention_mask=True)

input_features = inputs.input_features
attention_mask = inputs.attention_mask.to(torch.float32)

SEQ_LEN = input_features.shape[1]
FEATURE_DIM = input_features.shape[2]

print(f"\n✓ Input features shape: {input_features.shape}")
print(f"✓ Attention mask shape: {attention_mask.shape}")
print(f"✓ Model will be created with static shape: [1, {SEQ_LEN}, {FEATURE_DIM}]")

Full audio duration: 43.80 seconds
Optimized audio duration: 10.00 seconds
Sample rate: 16000 Hz

✓ Input features shape: torch.Size([1, 998, 128])
✓ Attention mask shape: torch.Size([1, 998])
✓ Model will be created with static shape: [1, 998, 128]


## 5. PyTorch Inference <a id="pytorch-inference"></a>

Run inference with PyTorch model to establish baseline accuracy.

In [48]:
# PyTorch inference
model.eval()
with torch.no_grad():
    pt_outputs = model(input_features, attention_mask=attention_mask.long())
    pt_logits = pt_outputs.logits.numpy()
    pt_ids = np.argmax(pt_logits, axis=-1)
    pt_transcription = tokenizer.batch_decode(pt_ids)[0]

print("PyTorch Inference Results:")
print(f"Transcription: {pt_transcription}")
print(f"Logits shape: {pt_logits.shape}")
print(f"Logits range: [{pt_logits.min():.2f}, {pt_logits.max():.2f}]")

PyTorch Inference Results:
Transcription: [EXAM TYPE] CT chest PE protocol {period} [INDICATION] 54-year-old female, shortness of breath, evaluate for PE {period}TECchHNIQe</s>
Logits shape: (1, 247, 512)
Logits range: [-26.49, 24.95]


## 6. Convert to OpenVINO FP16 <a id="convert-fp16"></a>

Convert the PyTorch model to OpenVINO IR format using `torch.export` and `ov.convert_model`.

In [49]:
import openvino as ov
import os

FP16_MODEL_PATH = Path("medasr_fp16.xml")

# Create model wrapper for clean export
class MedASRWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, input_features, attention_mask):
        outputs = self.model(input_features=input_features, attention_mask=attention_mask)
        return outputs.logits

wrapped_model = MedASRWrapper(model)
wrapped_model.eval()

print("Converting PyTorch model to OpenVINO IR...")
print(f"Input shape: {input_features.shape}")

with torch.no_grad():
    # Export using torch.export
    exported = torch.export.export(
        wrapped_model,
        (input_features, attention_mask)
    )
    print("✓ Model exported with torch.export")
    
    # Convert to OpenVINO
    ov_model = ov.convert_model(exported)
    
    # Reshape to static shape for optimal GPU performance
    ov_model.reshape({
        'input_features': [1, SEQ_LEN, FEATURE_DIM],
        'attention_mask': [1, SEQ_LEN]
    })
    print(f"✓ Model reshaped to static: [1, {SEQ_LEN}, {FEATURE_DIM}]")

# Save FP16 model (without FP16 compression to avoid GPU numerical issues)
ov.save_model(ov_model, FP16_MODEL_PATH, compress_to_fp16=False)

fp16_size = (os.path.getsize(FP16_MODEL_PATH) + os.path.getsize(FP16_MODEL_PATH.with_suffix('.bin'))) / 1024 / 1024
print(f"\n✓ FP16 model saved: {FP16_MODEL_PATH}")
print(f"✓ Model size: {fp16_size:.2f} MB")

# Verify model inputs
print("\nModel inputs:")
for inp in ov_model.inputs:
    print(f"  {inp.get_any_name()}: {inp.partial_shape}")

Converting PyTorch model to OpenVINO IR...
Input shape: torch.Size([1, 998, 128])
✓ Model exported with torch.export


/home/user/miniforge3/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


✓ Model reshaped to static: [1, 998, 128]

✓ FP16 model saved: medasr_fp16.xml
✓ Model size: 402.71 MB

Model inputs:
  input_features: [1,998,128]
  attention_mask: [1,998]


## 7. INT8 Quantization <a id="int8-quantization"></a>

Quantize the model to INT8 using NNCF with **real audio data** for calibration.

**Key Settings:**
- `ModelType.TRANSFORMER` - Optimized for transformer models
- Real audio calibration data - Better accuracy than random data
- `fast_bias_correction` - Faster quantization with good results

In [50]:
import nncf
from nncf import Dataset

INT8_MODEL_PATH = Path("medasr_int8.xml")

print("Preparing calibration data from real audio...")

# Create calibration data from the test audio with variations
calibration_data = []

# Use the real audio features as base
base_features = input_features.numpy().astype(np.float32)
base_mask = attention_mask.numpy().astype(np.float32)

# Add the original sample
calibration_data.append({
    'input_features': base_features,
    'attention_mask': base_mask
})

# Create variations with realistic audio augmentations
np.random.seed(42)
for i in range(99):  # Total 100 calibration samples
    # Add small realistic noise (simulates different recording conditions)
    noise_level = np.random.uniform(0.001, 0.02)
    noisy_features = base_features + np.random.randn(*base_features.shape).astype(np.float32) * noise_level
    
    # Slight volume variation
    volume_scale = np.random.uniform(0.8, 1.2)
    noisy_features = noisy_features * volume_scale
    
    calibration_data.append({
        'input_features': noisy_features,
        'attention_mask': base_mask.copy()
    })

print(f"✓ Created {len(calibration_data)} calibration samples")

# Create NNCF dataset
def transform_fn(data_item):
    return {
        'input_features': data_item['input_features'],
        'attention_mask': data_item['attention_mask']
    }

calibration_dataset = Dataset(calibration_data, transform_fn)

print("\nQuantizing to INT8 with TRANSFORMER preset...")
print("This may take a few minutes...")

quantized_model = nncf.quantize(
    model=ov_model,
    calibration_dataset=calibration_dataset,
    subset_size=min(100, len(calibration_data)),
    model_type=nncf.ModelType.TRANSFORMER,
    fast_bias_correction=True
)

print("✓ Quantization complete!")

# Verify INT8 model inputs
print("\nQuantized model inputs:")
for inp in quantized_model.inputs:
    print(f"  {inp.get_any_name()}: {inp.partial_shape}")

# Save INT8 model
ov.save_model(quantized_model, INT8_MODEL_PATH, compress_to_fp16=False)

int8_size = (os.path.getsize(INT8_MODEL_PATH) + os.path.getsize(INT8_MODEL_PATH.with_suffix('.bin'))) / 1024 / 1024
print(f"\n✓ INT8 model saved: {INT8_MODEL_PATH}")
print(f"✓ Model size: {int8_size:.2f} MB")
print(f"✓ Compression ratio: {fp16_size/int8_size:.2f}x")

Preparing calibration data from real audio...
✓ Created 100 calibration samples

Quantizing to INT8 with TRANSFORMER preset...
This may take a few minutes...


Output()

Output()

Output()

Output()

✓ Quantization complete!

Quantized model inputs:
  input_features: [1,998,128]
  attention_mask: [1,998]

✓ INT8 model saved: medasr_int8.xml
✓ Model size: 103.51 MB
✓ Compression ratio: 3.89x


In [51]:
# Display quantization statistics
op_types = {}
for op in quantized_model.get_ops():
    op_type = op.get_type_name()
    op_types[op_type] = op_types.get(op_type, 0) + 1

print("Quantized model statistics:")
print(f"  FakeQuantize ops: {op_types.get('FakeQuantize', 0)}")
print(f"  Convolution ops: {op_types.get('Convolution', 0)}")
print(f"  MatMul ops: {op_types.get('MatMul', 0)}")
print(f"  Total ops: {sum(op_types.values())}")

Quantized model statistics:
  FakeQuantize ops: 192
  Convolution ops: 37
  MatMul ops: 138
  Total ops: 4053


## 8. Accuracy Comparison <a id="accuracy-comparison"></a>

Compare accuracy of PyTorch, FP16, and INT8 models to ensure quantization quality.

In [52]:
import openvino as ov

print("="*70)
print("ACCURACY COMPARISON: PyTorch vs FP16 vs INT8")
print("="*70)

core = ov.Core()

# Prepare input data
np_features = input_features.numpy().astype(np.float32)
np_mask = attention_mask.numpy().astype(np.float32)

# Compile models for CPU (most accurate)
print("\nCompiling models for GPU...")
fp16_compiled = core.compile_model(FP16_MODEL_PATH, "GPU", {"PERFORMANCE_HINT": "LATENCY", "INFERENCE_PRECISION_HINT": "f32"})
int8_compiled = core.compile_model(INT8_MODEL_PATH, "GPU", {"PERFORMANCE_HINT": "LATENCY", "INFERENCE_PRECISION_HINT": "f32"})

# FP16 inference
fp16_out = fp16_compiled({"input_features": np_features, "attention_mask": np_mask})
fp16_logits = fp16_out[0]
fp16_ids = np.argmax(fp16_logits, axis=-1)
fp16_text = tokenizer.batch_decode(fp16_ids)[0]

# INT8 inference
int8_out = int8_compiled({"input_features": np_features, "attention_mask": np_mask})
int8_logits = int8_out[0]
int8_ids = np.argmax(int8_logits, axis=-1)
int8_text = tokenizer.batch_decode(int8_ids)[0]

print("\n--- Transcriptions ---")
print(f"PyTorch: {pt_transcription}")
print(f"FP16:    {fp16_text}")
print(f"INT8:    {int8_text}")

# Calculate accuracy metrics
def calculate_accuracy(ref_ids, hyp_ids):
    return np.mean(ref_ids == hyp_ids) * 100

fp16_vs_pytorch = calculate_accuracy(pt_ids, fp16_ids)
int8_vs_pytorch = calculate_accuracy(pt_ids, int8_ids)
int8_vs_fp16 = calculate_accuracy(fp16_ids, int8_ids)

print("\n--- Token Match Accuracy ---")
print(f"FP16 vs PyTorch: {fp16_vs_pytorch:.2f}%")
print(f"INT8 vs PyTorch: {int8_vs_pytorch:.2f}%")
print(f"INT8 vs FP16:    {int8_vs_fp16:.2f}%")

# Logit correlation
fp16_corr = np.corrcoef(pt_logits.flatten(), fp16_logits.flatten())[0, 1]
int8_corr = np.corrcoef(pt_logits.flatten(), int8_logits.flatten())[0, 1]

print("\n--- Logit Correlation ---")
print(f"FP16 vs PyTorch: {fp16_corr:.6f}")
print(f"INT8 vs PyTorch: {int8_corr:.6f}")

print("\n" + "="*70)
if fp16_vs_pytorch >= 99.0 and int8_vs_pytorch >= 95.0:
    print("✓ ACCURACY CHECK PASSED")
else:
    print("⚠ ACCURACY CHECK: Review results above")
print("="*70)

ACCURACY COMPARISON: PyTorch vs FP16 vs INT8

Compiling models for GPU...

--- Transcriptions ---
PyTorch: [EXAM TYPE] CT chest PE protocol {period} [INDICATION] 54-year-old female, shortness of breath, evaluate for PE {period}TECchHNIQe</s>
FP16:    [EXAM TYPE] CT chest PE protocol {period} [INDICATION] 54-year-old female, shortness of breath, evaluate for PE {period}TECchHNIQe</s>
INT8:    [EXAM TYPE] CT chest PE protocol {period} [INDICATION] 54-year-old female, shortness of breath, evaluate for PE {period}TECchHNiQe</s>

--- Token Match Accuracy ---
FP16 vs PyTorch: 100.00%
INT8 vs PyTorch: 98.38%
INT8 vs FP16:    98.38%

--- Logit Correlation ---
FP16 vs PyTorch: 1.000000
INT8 vs PyTorch: 0.996360

✓ ACCURACY CHECK PASSED


## 9. Performance Benchmarking <a id="benchmarking"></a>

Benchmark FP16 and INT8 models on GPU and CPU.


In [53]:
print("="*70)
print("PERFORMANCE BENCHMARKING")
print("="*70)

core = ov.Core()
available_devices = core.available_devices
print(f"Available devices: {available_devices}")

results = {}

# Benchmark configurations
devices_to_test = ["GPU", "CPU"] if "GPU" in available_devices else ["CPU"]

for device in devices_to_test:
    print(f"\n--- {device} Benchmarks ---")
    
    # Device-specific config
    
    config = {"PERFORMANCE_HINT": "LATENCY"}
    if device == "GPU":
        config["INFERENCE_PRECISION_HINT"] = "f32"
        
    
    # FP16 benchmark
    fp16_model = core.compile_model(FP16_MODEL_PATH, device, config)
    
    # Warmup
    for _ in range(10):
        fp16_model({"input_features": np_features, "attention_mask": np_mask})
    
    # Benchmark
    fp16_latencies = []
    for _ in range(100):
        start = time.time()
        fp16_model({"input_features": np_features, "attention_mask": np_mask})
        fp16_latencies.append((time.time() - start) * 1000)
    
    fp16_median = np.median(fp16_latencies)
    fp16_min = np.min(fp16_latencies)
    
    # INT8 benchmark
    int8_model = core.compile_model(INT8_MODEL_PATH, device, config)
    
    # Warmup
    for _ in range(10):
        int8_model({"input_features": np_features, "attention_mask": np_mask})
    
    # Benchmark
    int8_latencies = []
    for _ in range(100):
        start = time.time()
        int8_model({"input_features": np_features, "attention_mask": np_mask})
        int8_latencies.append((time.time() - start) * 1000)
    
    int8_median = np.median(int8_latencies)
    int8_min = np.min(int8_latencies)
    
    speedup = fp16_median / int8_median
    
    print(f"FP16: {fp16_median:.2f}ms (min: {fp16_min:.2f}ms)")
    print(f"INT8: {int8_median:.2f}ms (min: {int8_min:.2f}ms)")
    print(f"Speedup: {speedup:.2f}x")
    
    results[device] = {
        "fp16_median_ms": fp16_median,
        "int8_median_ms": int8_median,
        "speedup": speedup
    }

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"\nModel sizes:")
print(f"  FP16: {fp16_size:.2f} MB")
print(f"  INT8: {int8_size:.2f} MB")
print(f"  Compression: {fp16_size/int8_size:.2f}x")

print(f"\nAccuracy (vs PyTorch):")
print(f"  FP16: {fp16_vs_pytorch:.2f}%")
print(f"  INT8: {int8_vs_pytorch:.2f}%")


print("="*70)

PERFORMANCE BENCHMARKING
Available devices: ['CPU', 'GPU', 'NPU']

--- GPU Benchmarks ---
FP16: 38.78ms (min: 37.15ms)
INT8: 6.57ms (min: 6.41ms)
Speedup: 5.90x

--- CPU Benchmarks ---
FP16: 140.30ms (min: 138.57ms)
INT8: 45.81ms (min: 45.39ms)
Speedup: 3.06x

SUMMARY

Model sizes:
  FP16: 402.71 MB
  INT8: 103.51 MB
  Compression: 3.89x

Accuracy (vs PyTorch):
  FP16: 100.00%
  INT8: 98.38%


In [54]:
# Save test data for benchmark script
print("Saving test data for benchmark scripts...")

np.save('medasr_input_features_10s.npy', np_features)
np.save('medasr_attention_mask_10s.npy', np_mask)

# Create 20s and 30s test data by padding
features_20s = np.pad(np_features, ((0,0), (0, SEQ_LEN), (0,0)), mode='edge')
mask_20s = np.pad(np_mask, ((0,0), (0, SEQ_LEN)), mode='constant', constant_values=0)
np.save('medasr_input_features_20s.npy', features_20s)
np.save('medasr_attention_mask_20s.npy', mask_20s)

features_30s = np.pad(np_features, ((0,0), (0, SEQ_LEN*2), (0,0)), mode='edge')
mask_30s = np.pad(np_mask, ((0,0), (0, SEQ_LEN*2)), mode='constant', constant_values=0)
np.save('medasr_input_features_30s.npy', features_30s)
np.save('medasr_attention_mask_30s.npy', mask_30s)

print(f"✓ 10s data: {np_features.shape}")
print(f"✓ 20s data: {features_20s.shape}")  
print(f"✓ 30s data: {features_30s.shape}")
print("\nFiles saved for benchmark_medasr_durations.py")

Saving test data for benchmark scripts...
✓ 10s data: (1, 998, 128)
✓ 20s data: (1, 1996, 128)
✓ 30s data: (1, 2994, 128)

Files saved for benchmark_medasr_durations.py


## Summary

This notebook created optimized OpenVINO models for MedASR:

**Generated Models:**
- `medasr_fp16.xml` - FP16 model for CPU/GPU inference
- `medasr_int8.xml` - INT8 quantized model with ~2x compression

**Key Results:**
- Static model shape: `[1, 998, 128]` (optimized for 10s audio)
- INT8 quantization using real audio calibration data
- GPU acceleration with LATENCY performance hint

